### 一、以下是单推理谓词的结构优先的整体流程示例

#### 1. 将 CSV 数据转换为 GraphLib 格式，这个格式中边是带标签的:

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os
import re
import sys
import math
import time
import tempfile
import traceback
from typing import List, Dict, Tuple
import pandas as pd
import numpy as np
import json
project_root = "/home/wangshuo/projects/Neo4j_Exp"
if project_root not in sys.path:
    sys.path.append(project_root)
from pythonProject.src.Structure_first.fastest_pipeline import FastestGraphConverter, FastestEstimateMerger
from pythonProject.src.Structure_first.graph_sample import FastestRunner
from pythonProject.src.Structure_first.precision_submatching import ExactSubgraphMatcher
from pythonProject.src.Structure_first.proxy_sample import ProxyStratifiedSampler, compute_T_true


# 一级测试数据集
datasets_name = "parler_data"
# 一级数据集下根据查询和图结构的差异划分的子测试数据集
# dataset_name = "dataset_three"
dataset_name = "dataset_two"
# 原始CSV数据路径
CSV_BASE_DIR = f"/home/wangshuo/resource/datasets/{datasets_name}/{dataset_name}/csv_data"
# 转换后GraphLib数据存放路径
Graph_Lib_Dir = f"/home/wangshuo/resource/datasets/{datasets_name}/{dataset_name}/data_graph"

# converter = FastestGraphConverter(CSV_BASE_DIR,Graph_Lib_Dir)
# converter.run_without_author_user_post()
# converter.remove_edge_labels()

#### 2.1同时调用准确的子图匹配算法得到真实结果的基数，这些准确子图匹配算法只能处理边不带标签的图

##### 2.1.1 首先基于原来边带标签的图，精简成边不带标签的图

In [ ]:
# converter = FastestGraphConverter(CSV_BASE_DIR,Graph_Lib_Dir)
# converter.simplify_graph_merge_edges_update_degree(input_path=Graph_Lib_Dir+'/parler.graph',
#                                                    output_path='/home/wangshuo/resource/datasets/parler_data/dataset_one/ground_truth/data_graph/parler.graph')

##### 2.1.2 然后对精简后的数据图调用准确子图匹配算法进行匹配，这里得到的GT对应的查询图，应该与Fastest对应的查询图等价（就算边没有标签也没有影响，且两点不存在多条边）

                                        六千万基数： 准确匹配需要2min左右
                                        1.2亿基数： 准确匹配需要4min左右
                                        2.4亿基数： 准确匹配需要8min左右
                                        10亿基数：  准确匹配需要40min左右

In [ ]:
matcher = ExactSubgraphMatcher(
    exe_path="/home/wangshuo/projects/SubgraphMatching/build/matching/SubgraphMatching.out",
    default_args=["-filter", "GQL", "-order", "GQL", "-engine", "LFTJ", "-num", "MAX"],
    timeout=3000,
)

matcher.run_batch(
    data_graph=f"/home/wangshuo/resource/datasets/parler_data/{dataset_name}/data_graph/parler.graph",
    query_graph_dir=f"/home/wangshuo/resource/datasets/parler_data/{dataset_name}/query_graph",
    output_dir=f"/home/wangshuo/resource/datasets/parler_data/{dataset_name}/ground_truth",
)

#### 2.2 基于GraphLib数据，调用Fastest进行树采样或图采样，得到特定标签下所有或部分点的估计值:
（1) 对于随机生成的查询可以先调用Fastest估计基数，选择基数合适的查询作为实验对象 --- 对于parler_ans.txt中的数据可以先随机生成
（2）正常流畅下，需要先调用精确的子图匹配算法，得到真实基数结果存放到parler_ans.txt，然后再调用Fastest进行估计

In [3]:
# 初始化 Runner
runner = FastestRunner(build_dir="/home/wangshuo/projects/FaSTest-main/build")

# dataset_name = "dataset_three"
current_budget = 20000
infer_label = 1  # 对应 Post 节点的标签

# 调用 run 方法
code, output = runner.run(
    dataset=dataset_name,
    root_label=infer_label,           # 必须指定推理节点的标签
    sample_budget=current_budget,     # 设置预算
    # estimate_with_predicate=True      # <--- 开启单推理谓词模式
    estimate_with_predicate=False      # <--- 关闭单推理谓词模式
)

🚀 正在运行: /home/wangshuo/projects/FaSTest-main/build/Fastest -d dataset_two --ROOT_LABEL 1 --SAMPLE_BUDGET 20000

[CLI] Root label set to 1
[CLI] Sample budget set to 20000
[info] SAMPLE_BUDGET:20000
[Info] Cleared old estimate file: /home/wangshuo/resource/datasets/parler_data/dataset_two/results/in_estimateW_result.txt
[info] args:0x7ffdc5e76758
/home/wangshuo/resource/datasets/parler_data/dataset_two/ground_truth/parler_ans.txt
[Info] Reading core labels configuration from: /home/wangshuo/resource/datasets/parler_data/dataset_two/data_graph/user_custom_labels.txt
[Info] Core labels configuration file not found. Running in global estimation mode only.
[infor] root label1
Constructing Candidate Space: 12 22
[info] Query Size:65

Start Processing (Global) /home/wangshuo/resource/datasets/parler_data/dataset_two/query_graph/query_cycle_4_0.graph
[Info] Using user-specified root label 1 -> query node 1
[Check] Sum(node_est) = 4.93824e+06, est_total = 4.93824e+06, ratio = 1
[INFO] sv_out_pa

#### 2.3 生成推理节点文件infer_node.txt：
(1)得到准确的子图匹配结果后，要计算最终的真实的答案ST_IF_Truth，下直称T_truth,需要知道每个查询图中，代表推理谓词节点的顶点编号
(2)一个查询图的格式如下，三个顶点按照id依次编号为u0,u1,u2,接下来我想读取/home/wangshuo/resource/datasets/parler_data/dataset_one/query_graph文件夹下的所有查询文件，将标签为1的顶点所对应的标号保存到infer_node.txt，以下面查询为例要保存u1到文件中，
                                            t 3 3
                                            v 0 0 2
                                            v 1 1 2
                                            v 2 2 2
                                            e 1 2
                                            e 2 0
                                            e 1 0

In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

"""
extract_infer_node.py
从 query_graph 文件夹下的所有查询 .graph 文件中，
提取 label==1 的顶点编号，写入 infer_node.txt。
"""

import os

def extract_infer_node(dataset_name: str) -> str:
    """
    从指定数据集的 query_graph 文件夹下提取所有查询图中 label==1 的节点，
    并保存到 data_graph/infer_node.txt 文件中。

    参数:
        dataset_name (str): 数据集名称，如 "parler_data" 或 "dataset_two"

    返回:
        str: 输出文件路径
    """

    # === 配置路径 ===
    query_dir = f"/home/wangshuo/resource/datasets/parler_data/{dataset_name}/query_graph"
    out_path = os.path.join(
        f"/home/wangshuo/resource/datasets/parler_data/{dataset_name}/data_graph",
        "infer_node.txt"
    )

    # === 收集所有 .graph 文件 ===
    graph_files = []
    for root, _, files in os.walk(query_dir):
        for f in files:
            if f.endswith(".graph"):
                graph_files.append(os.path.join(root, f))
    graph_files.sort()

    print(f"[INFO] Found {len(graph_files)} query graph files under {query_dir}")

    # === 解析每个文件，提取 label==1 的节点编号 ===
    results = []
    for path in graph_files:
        with open(path, "r") as f:
            for line in f:
                line = line.strip()
                if not line or not line.startswith("v "):
                    continue
                parts = line.split()
                if len(parts) >= 3:
                    vid = parts[1]
                    label = parts[2]
                    if label == "1":
                        results.append(f"u{vid}")
                        break  # 每个查询只有一个 label==1，提取后退出

    # === 写入输出文件 ===
    os.makedirs(os.path.dirname(out_path), exist_ok=True)
    with open(out_path, "w") as fout:
        for r in results:
            fout.write(r + "\n")

    print(f"[DONE] Saved {len(results)} infer nodes to {out_path}")
    return out_path


# === 如果直接运行此脚本 ===
if __name__ == "__main__":
    dataset_name = "dataset_one"  # ✅ 修改为你的数据集名
    extract_infer_node(dataset_name)


### 3 - 4:
"""
处理 multi-query 的 Fastest 输出(in_estimateW_result.txt)：
  - 解析多个 Query 的每节点估计值（支持多个 Query 块）
  - 根据 INFER_NODE_FILE 中每行指定的 gt_match_col（例如 u1,u2）按顺序对应每个 Query
  - 将每个 Query 的 estimate 列并入 post_with_estimate.csv（列名 estimate__<query_basename>）
  - 针对每个 Query 用 ProxyStratifiedSampler 做评估（使用 compute_T_true 得到 T_true）
  - 输出 summary CSV / TXT

"""

#### 3.处理 Fastest 输出结果，将估计值与目标节点的文件合并

In [4]:

from pythonProject.src.Structure_first.proxy_sample import ProxyStratifiedSampler, compute_T_true

# ===========================
# = 配置区域（按需修改） =
# ===========================
# dataset_name = 'dataset_two'
print(dataset_name)

# Fastest对所有查询图的估计结果文件保存路径
SV_FILE = f"/home/wangshuo/resource/datasets/parler_data/{dataset_name}/results/in_estimateW_result.txt"
# 用户指定每个查询图中推理谓词节点
INFER_NODE_FILE = f"/home/wangshuo/resource/datasets/parler_data/{dataset_name}/data_graph/infer_node.txt"
# 图中节点 ID 与原始CSV文件中数据id的映射关系文件
IDMAP_FILE = f"/home/wangshuo/resource/datasets/parler_data/{dataset_name}/data_graph/id_mapping.csv"
# 原始 post.csv 文件路径
POST_CSV = f"/home/wangshuo/resource/datasets/parler_data/{dataset_name}/csv_data/post.csv"
# 存放真实结果的目录
GT_RESULT_DIR = f"/home/wangshuo/resource/datasets/parler_data/{dataset_name}/ground_truth/structure_result"
# 存放本原型系统结果的目录
OUTPUT_DIR = f"/home/wangshuo/resource/datasets/parler_data/{dataset_name}/results"
# 输出的包含每个查询post估计值w（x）的 post CSV 文件路径
POST_WITH_ESTIMATE_CSV = os.path.join(OUTPUT_DIR, "post_with_estimate.csv")
# 各方法结果的总结文件路径
SUMMARY_CSV = os.path.join(OUTPUT_DIR, "results_summary.csv")
SUMMARY_TXT = os.path.join(OUTPUT_DIR, "results_summary.txt")
# 是否保留临时 CSV（用于调试）
KEEP_TEMP = False  
# ===========================

os.makedirs(OUTPUT_DIR, exist_ok=True)

# ---------------------------
# Main pipeline
# ---------------------------



dataset_one


In [5]:

print("[BEGIN] multi-query pipeline")

merger = FastestEstimateMerger(
    sv_file=SV_FILE,
    map_file=IDMAP_FILE,
    post_file=POST_CSV,
    output_file=POST_WITH_ESTIMATE_CSV
)
# 1) parse sv multi
sv_df = merger.parse_sv_multi(SV_FILE)
if sv_df is None or sv_df.empty:
    print("[ERROR] No sv records parsed. Exiting.")

# 2) read infer node list
infer_nodes = merger.read_infer_node_list(INFER_NODE_FILE)

# 3) build post_with_estimate.csv
merged_df = merger.build_post_with_estimates(sv_df=sv_df, idmap_file=IDMAP_FILE, post_csv=POST_CSV,
                                     out_csv=POST_WITH_ESTIMATE_CSV)



[BEGIN] multi-query pipeline
[INFO] parse_sv_multi: parsed 222994 node records across 16 queries.
[INFO] read_infer_node_list: loaded 16 infer nodes (first 10): ['u1', 'u2', 'u2', 'u1', 'u2', 'u1', 'u2', 'u2', 'u1', 'u2']
[INFO] build_post_with_estimates: wrote /home/wangshuo/resource/datasets/parler_data/dataset_one/results/post_with_estimate.csv, total rows=24082


### 4. 解析 Fastest 输出结果： 基于单推理谓词也就是root_label各个节点的权重估计值进行proxy指导采样

In [9]:
from pythonProject.src.Structure_first.proxy_sample import compute_T_true_polars
# 定义统计结果保存路径 (确保是全局变量或在函数内定义)
SAMPLED_COUNT_FILE = os.path.join(OUTPUT_DIR, "efficiency/sampled_node_count.csv")

def save_node_counts(records: List[Dict]):
    """辅助函数：将节点计数追加到 CSV"""
    if not records: return
    df = pd.DataFrame(records)
    header = not os.path.exists(SAMPLED_COUNT_FILE)
    try:
        df.to_csv(SAMPLED_COUNT_FILE, mode='a', index=False, header=header)
    except Exception as e:
        print(f"[错误] 写入节点统计失败: {e}")

# 评估参数
RUN_TIMES = 1  # 每种方法重复次数
# ---------------------------
# Step D: 对每个 Query 计算 T_true 并评估
# ---------------------------
print(dataset_name)
def evaluate_queries(
    merged_df: pd.DataFrame,
    sv_df: pd.DataFrame,
    infer_nodes: List[str],
    idmap_file: str,
    post_csv: str,
    gt_result_dir: str,
    output_dir: str,
    run_times: int = 10,
    new_workload: bool = False,  # ✅ 新增：是否重新计算所有 T_true
    proxy_model = 'ML1_proxy2b1_probability',
    T_true_cache_file: str = f"/home/wangshuo/resource/datasets/parler_data/{dataset_name}/ground_truth/T_true.txt"  # ✅ 新增：缓存文件路径    
)  -> Tuple[pd.DataFrame, pd.DataFrame]:
    """
    对每个 query（按 sv_df 中 query_index 出现顺序）：
      - 使用 infer_nodes 对应的 gt_match_col 计算 T_true（若找不到 GT 文件则 T_true=0）
      - 将对应的 estimate__col 作为 'estimate' 列写入临时 csv
      - 用 ProxyStratifiedSampler 执行四种方法（每种重复 run_times 次），取均值与 std
      - 返回 summary dataframe
    """
    rows = []
    txt_lines = []
    all_node_stats_records = []
    print(f"[BEGIN] evaluate_queries with proxy_model={proxy_model}")
    # ✅ Step 1: 读取/初始化 T_true 缓存
    if os.path.exists(T_true_cache_file) and not new_workload:
        with open(T_true_cache_file, "r") as f:
            try:
                T_true_cache = json.load(f)
                print(f"[INFO] 已加载缓存的 T_true，共 {len(T_true_cache)} 条。")
            except json.JSONDecodeError:
                print(f"[WARN] {T_true_cache_file} 格式错误，将重新计算所有 T_true。")
                T_true_cache = {}
    else:
        T_true_cache = {}

    # order queries by index
    q_order = sv_df[["query_index", "query_basename"]].drop_duplicates().sort_values("query_index")
    q_order = q_order.reset_index(drop=True)

    n_queries = len(q_order)
    if len(infer_nodes) < n_queries:
        print(f"[WARN] infer_nodes length {len(infer_nodes)} < number of parsed queries {n_queries}. We'll map as many as available and default 'u1' for missing.")
    
    # ✅ Step 2: 遍历每个 query
    for i, r in q_order.iterrows():
        qi = int(r["query_index"])
        qbase = r["query_basename"]
        colname = f"estimate__{qi}__{qbase}"
        print("\n" + "="*60)
        print(f"[STEP] Query index={qi}, basename={qbase}, estimate column={colname}")

        # choose gt_match_col from infer_nodes by index (if available)
        gt_match_col = infer_nodes[i] if i < len(infer_nodes) else "u1"
        print(f"[INFO] Using gt_match_col = {gt_match_col} for query #{qi}")

        # ✅ Step 3: 读取或计算 T_true
        if (not new_workload) and (qbase in T_true_cache):
            T_true = float(T_true_cache[qbase])
            print(f"[CACHE] 读取缓存 T_true={T_true:.4f} for {qbase}")
        else:
            # locate GT file for this query
            # attempt a few plausible filenames in gt_result_dir
            candidates = [
                os.path.join(gt_result_dir, f"{qbase}_matches.csv"),
                os.path.join(gt_result_dir, f"{qbase}.graph_matches.csv"),
                os.path.join(gt_result_dir, f"{qbase}.matches.csv"),
                os.path.join(gt_result_dir, qbase),
            ]
            gt_path = None
            for p in candidates:
                if p and os.path.exists(p):
                    gt_path = p
                    break
            if gt_path is None:
                # try to find any file in gt_result_dir that contains the qbase as substring
                for fname in os.listdir(gt_result_dir):
                    if qbase in fname:
                        cand = os.path.join(gt_result_dir, fname)
                        if os.path.isfile(cand):
                            gt_path = cand
                            break
            if gt_path is None:
                print(f"[WARN] Ground-truth file for query {qbase} not found in {gt_result_dir}. Will use T_true=0.")
                T_true = 0.0
            else:
                try:
                    # compute_T_true should be available (user provided)
                    T_true = compute_T_true_polars(
                        gt_path=gt_path,
                        id_mapping_path=idmap_file,
                        post_csv_path=post_csv,
                        gt_match_col=gt_match_col,
                        prob_col="ML1_oracle1_probability",
                        prob_threshold=0.5
                    )
                except Exception as e:
                    print(f"[ERROR] compute_T_true_polars failed for {qbase} with gt_match_col={gt_match_col}: {e}")
                    traceback.print_exc()
                    T_true = 0.0
            # ✅ 写入缓存
            T_true_cache[qbase] = float(T_true)
            with open(T_true_cache_file, "w") as f:
                json.dump(T_true_cache, f, indent=2)
            print(f"[CACHE] 已更新缓存: {qbase} -> {T_true:.4f}")
            
        # Step 4: 临时 CSV prepare temp CSV for sampler: copy merged_df and set 'estimate' = that col
        if colname not in merged_df.columns:
            print(f"[WARN] estimate column {colname} not found in merged_df. Using zeros.")
            tmp_df = merged_df.copy()
            tmp_df["estimate"] = 0.0
        else:
            tmp_df = merged_df.copy()
            tmp_df["estimate"] = tmp_df[colname].astype(float).fillna(0.0)

        # create temp csv file
        tmp_csv = os.path.join(output_dir, f"tmp_post_with_estimate_q{qi}__{qbase}.csv")
        tmp_df.to_csv(tmp_csv, index=False)
        if not KEEP_TEMP:
            remove_tmp = True
        else:
            remove_tmp = False

        # instantiate sampler (user-provided)
        # sampler = ProxyStratifiedSampler(csv_path=tmp_csv, T_true=T_true,proxy_model=proxy_model)
        sampler = ProxyStratifiedSampler(csv_path=tmp_csv, T_true=T_true,is_multi_predicate=False, # 明确告知这是单谓词场景
                                         post_proxy=proxy_model,     # 将 proxy_model 赋值给 post_proxy
                                        total_budget_frac=0.1,
                                         c_stage=0.15
                                        )

        methods = {
            # ✅ 新增三种 baseline
            "baseline_uniform": sampler.run_baseline_uniform,
            # "baseline_proxy": sampler.run_baseline_proxy,
            "baseline_proxy_a": sampler.run_baseline_proxy_a,
            # "baseline_proxy_a_unbiased": sampler.run_baseline_proxy_a_unbiased,
            "baseline_graph_only": sampler.run_baseline_graph_only,
            # "baseline_a": sampler.run_baseline_a,                   # proportional to a
            # "baseline_proxy_mul_a": sampler.run_baseline_proxy_mul_a, # proportional to proxy * a
            # 原有四种分层采样法
            "proxy_importance": sampler.run_proxy_importance,
            "proxy_mab": sampler.run_mab_sampling,
            # "proxy_uniform": sampler.run_proxy_uniform,
            "proxyE_importance": sampler.run_proxyE_importance,
            # "proxyE_uniform": sampler.run_proxyE_uniform
        }

        for mname, func in methods.items():
            T_list = []
            Q_list = []
            post_cnt_list = []
            comment_cnt_list = [] # 单谓词下通常为0
            # +++ 新增：收集 pi 统计信息 +++
            pi_mean_list = []
            pi_min_list = []
            pi_max_list = []
            
            print(f"\n--- Running {mname} for {run_times} times ---")
            for t in range(run_times):
                try:
                    out = func()
                    T_hat = float(out.get("T_hat", 0.0))
                    Qerror = float(out.get("Qerror", 1.0))
                    T_list.append(T_hat)
                    Q_list.append(Qerror)
                    # ✅ 实时打印每次结果
                    # print(f"[{mname} | run {t+1}/{run_times}]  T_hat={T_hat:.4f},  Qerror={Qerror:.6f}")
                    post_cnt_list.append(out.get("n_post", 0))
                    comment_cnt_list.append(out.get("n_comment", 0))
                    pi_mean_list.append(out.get("pi_mean", 0.0))
                    pi_min_list.append(out.get("pi_min", 0.0))
                    pi_max_list.append(out.get("pi_max", 0.0))
                except Exception as e:
                    print(f"❌ {mname} 第 {t+1} 次执行失败: {repr(e)}")
                    traceback.print_exc()
                    T_list.append(0.0)
                    Q_list.append(1.0)
                    list.append(0)
                    comment_cnt_list.append(0)

            # # 计算均值与标准差
            # T_mean = float(np.mean(T_list))
            # T_std = float(np.std(T_list))
            # Q_mean = float(np.mean(Q_list))
            # Q_std = float(np.std(Q_list))
            # 去掉最高值和最低值计算均值和标准差
            def trimmed_mean_std(lst):
                if len(lst) <= 2:
                    return np.mean(lst), np.std(lst)
                sorted_lst = sorted(lst)
                trimmed = sorted_lst[1:-1]
                return np.mean(trimmed), np.std(trimmed)
            
            T_mean, T_std = trimmed_mean_std(T_list)
            Q_mean, Q_std = trimmed_mean_std(Q_list)
            avg_post = int(np.mean(post_cnt_list))
            avg_comment = int(np.mean(comment_cnt_list))
            
            avg_pi_mean = np.mean(pi_mean_list)
            avg_pi_min = np.mean(pi_min_list)
            avg_pi_max = np.mean(pi_max_list)

            print(f"✅ {mname} 平均结果: T_hat={T_mean:.4f}±{T_std:.4f},  Qerror={Q_mean:.6f}±{Q_std:.6f}")
            print(f"   Samples(P/C)={avg_post}/{avg_comment} | Pi(Mean/Min/Max)={avg_pi_mean:.4f} / {avg_pi_min:.4f} / {avg_pi_max:.4f}")
            # 写入结果表
            rows.append({
                "query_index": qi,
                "query_basename": qbase,
                "gt_match_col": gt_match_col,
                "T_true": float(T_true),
                "method": mname,
                "T_hat_mean": T_mean,
                "T_hat_std": T_std,
                "Qerror_mean": Q_mean,
                "Qerror_std": Q_std
            })

            # 同时写入 TXT 文本（summary）
            txt_lines.append(
                f"{qbase} {gt_match_col} {mname} "
                f"T_hat={T_mean:.6f}±{T_std:.6f} "
                f"Qerror={Q_mean:.6f}±{Q_std:.6f}"
            )
            # 记录节点采样统计
            all_node_stats_records.append({
                "query_name": qbase,
                "method": mname,
                "post_sampled_cnt": avg_post,
                "comment_sampled_cnt": avg_comment
            })

        # cleanup tmp csv
        if remove_tmp:
            try:
                os.remove(tmp_csv)
            except Exception:
                pass

    # write summary files
    df_summary = pd.DataFrame(rows)
    df_summary.to_csv(SUMMARY_CSV, index=False)
    with open(SUMMARY_TXT, "w") as f:
        for ln in txt_lines:
            f.write(ln + "\n")
    if all_node_stats_records:
        save_node_counts(all_node_stats_records)
        print(f"[INFO] 已保存节点采样统计到 {SAMPLED_COUNT_FILE}")
    print(f"[INFO] evaluate_queries: wrote summary to {SUMMARY_CSV} and {SUMMARY_TXT}")
    return df_summary,all_node_stats_records


# 4) evaluate each query
summary_df, df_node_stats = evaluate_queries(
    merged_df=merged_df,
    sv_df=sv_df,
    infer_nodes=infer_nodes,
    idmap_file=IDMAP_FILE,
    post_csv=POST_CSV,
    gt_result_dir=GT_RESULT_DIR,
    output_dir=OUTPUT_DIR,
    run_times=RUN_TIMES,
    proxy_model='ML1_proxy2b1_probability',
)

print("[END] pipeline completed")
if summary_df is not None:
    print(summary_df.head())


dataset_one
[BEGIN] evaluate_queries with proxy_model=ML1_proxy2b1_probability
[INFO] 已加载缓存的 T_true，共 21 条。

[STEP] Query index=0, basename=query_dense_1_1.graph, estimate column=estimate__0__query_dense_1_1.graph
[INFO] Using gt_match_col = u1 for query #0
[CACHE] 读取缓存 T_true=16656.0000 for query_dense_1_1.graph
[Check_total_budget_frac] 0.1
[Check_c_stage] 0.15
[Check_K] 5

--- Running baseline_uniform for 1 times ---
✅ baseline_uniform 平均结果: T_hat=19778.1417±0.0000,  Qerror=0.187448±0.000000
   Samples(P/C)=994/0 | Pi(Mean/Min/Max)=0.0999 / 0.0999 / 0.0999

--- Running baseline_proxy_a for 1 times ---
✅ baseline_proxy_a 平均结果: T_hat=18393.8465±0.0000,  Qerror=0.104338±0.000000
   Samples(P/C)=994/0 | Pi(Mean/Min/Max)=0.4311 / 0.0075 / 1.0000

--- Running baseline_graph_only for 1 times ---
✅ baseline_graph_only 平均结果: T_hat=16694.5723±0.0000,  Qerror=0.002316±0.000000
   Samples(P/C)=9948/0 | Pi(Mean/Min/Max)=1.0000 / 1.0000 / 1.0000

--- Running proxy_importance for 1 times ---
[Chec

# ---------------------------------------------------------------------------------------------------------------------------------------------------------
#  下面就不属于原型系统的内容了，是各种处理脚本

### 合并基线方法和my method方法的输出结果

In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

"""
merge_results_summary.py
将 results_summary.csv 与 results_summary_baseline.csv 合并成统一文件，
并重新编码 query_index，确保唯一性。
"""

import pandas as pd

dataset_name = 'dataset_three'
# ====== 输入输出路径 ======
base_csv = f"/home/wangshuo/resource/datasets/parler_data/{dataset_name}/results/results_summary_method.csv"              # 原四种方法
baseline_csv = f"/home/wangshuo/resource/datasets/parler_data/{dataset_name}/results/results_summary.csv" # 新增三种 baseline
out_csv = f"/home/wangshuo/resource/datasets/parler_data/{dataset_name}/results/results_summary_merged.csv"        # 输出合并结果
# ==========================

# 读取文件
df_base = pd.read_csv(base_csv)
df_baseline = pd.read_csv(baseline_csv)

# 检查列是否一致
assert set(df_base.columns) == set(df_baseline.columns), "两份 CSV 列名不一致，请检查"

# === 关键处理：统一 query_index 编码方式 ===
# 方案1（推荐）：
#   用 query_basename 来重新分配 query_index，使得每个 query_basename 的 index 唯一。
#   这样可以防止不同文件的 index 冲突。

# 合并后重编号
df_all = pd.concat([df_base, df_baseline], ignore_index=True)

# 根据 query_basename 重新编号（保证同一 query 文件的编号一致）
basename_to_idx = {name: i for i, name in enumerate(sorted(df_all["query_basename"].unique()))}
df_all["query_index"] = df_all["query_basename"].map(basename_to_idx)

# 检查：确认每个 query_basename 映射唯一编号
assert df_all.groupby("query_basename")["query_index"].nunique().eq(1).all(), "存在 query_index 编码重复问题"

# === 生成带符号误差列，便于后续绘图 ===
df_all["Qerror_signed"] = (df_all["T_hat_mean"] - df_all["T_true"]) / df_all["T_true"]

# === 保存结果 ===
df_all.to_csv(out_csv, index=False)
print(f"[OK] 已合并 {len(df_all)} 条记录，保存至 {out_csv}")

# === 可选：打印合并后的方法统计 ===
print("\n[INFO] 各方法条目数：")
print(df_all["method"].value_counts())

print("\n[INFO] 各查询类型统计：")
print(df_all["query_basename"].apply(lambda x: x.split('_')[1]).value_counts())


### 读取下面csv文件，将 query_basename 和 T_true 对应保存到 T_true.txt 中，按json格式保存

In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

"""
extract_T_true_to_json.py
从结果 CSV 文件中提取 (query_basename, T_true)，
并以 JSON 格式保存到 T_true.txt。
"""

import pandas as pd
import json

# ======== 配置 ========
csv_file = "/home/wangshuo/resource/datasets/parler_data/dataset_one/results/results_summary_merged.csv"   # 你的 CSV 文件路径
output_file = "/home/wangshuo/resource/datasets/parler_data/dataset_one/ground_truth/T_true.txt"                  # 输出 JSON 文件路径
# =======================

# 读取 CSV
df = pd.read_csv(csv_file)

# 提取唯一的 query_basename 与 T_true 对应关系
ttrue_dict = (
    df.groupby("query_basename")["T_true"]
    .first()   # 每个 query_basename 保留第一个 T_true
    .to_dict()
)

# 写入 JSON 文件（缩进格式美观）
with open(output_file, "w", encoding="utf-8") as f:
    json.dump(ttrue_dict, f, indent=2, ensure_ascii=False)

print(f"[INFO] 已保存 {len(ttrue_dict)} 条 T_true 记录到 {output_file}")


V1.0:  单谓词单查询图的结构优先整体流程示例，包括：

In [ ]:
import os

import numpy as np

from pythonProject.src.Structure_first.fastest_pipeline import FastestGraphConverter, FastestEstimateMerger
from pythonProject.src.Structure_first.graph_sample import FastestRunner
from pythonProject.src.Structure_first.precision_submatching import ExactSubgraphMatcher
from pythonProject.src.Structure_first.proxy_sample import ProxyStratifiedSampler, compute_T_true

# 以下是单推理谓词的结构优先的整体流程示例

if __name__ == "__main__":
    datasets_name = "parler_data"
    dataset_name = "dataset_one"
    CSV_BASE_DIR = f"/home/wangshuo/resource/datasets/{datasets_name}/{dataset_name}/csv_data"
    Graph_Lib_Dir = f"/home/wangshuo/resource/datasets/{datasets_name}/{dataset_name}/data_graph"
    # # 1. 将 CSV 数据转换为 GraphLib 格式，这个格式中边是带标签的
    # converter = FastestGraphConverter(CSV_BASE_DIR,Graph_Lib_Dir)
    # converter.run()

    # # 2.基于GraphLib数据，调用Fastest进行树采样或图采样，得到特定标签下所有或部分点的估计值
    # runner = FastestRunner(
    #     build_dir="/home/wangshuo/projects/FaSTest-main/build"
    # )
    # # 默认执行 ./Fastest -d parler --ROOT_LABEL 1 (表示推理谓词所在节点的标签)
    # code, output = runner.run(dataset="parler", root_label=1)
    # 2.1同时调用准确的子图匹配算法得到真实结果的基数，这些准确子图匹配算法只能处理边不带标签的图
    # # 2.1.1 所以首先基于原来边带标签的图，精简成边不带标签的图
    # converter = FastestGraphConverter(CSV_BASE_DIR,Graph_Lib_Dir)
    # converter.simplify_graph_merge_edges_update_degree(input_path=Graph_Lib_Dir+'/parler.graph',
    #                                                    output_path='/home/wangshuo/resource/datasets/parler_data/dataset_one/ground_truth/data_graph/parler.graph')
    # 2.1.2 然后对精简后的数据图调用准确子图匹配算法进行匹配，这里得到的GT对应的查询图，应该与Fastest对应的查询图等价（就算边没有标签也没有影响，且两点不存在多条边）
    # matcher = ExactSubgraphMatcher(
    #     exe_path="/home/wangshuo/projects/SubgraphMatching/build/matching/SubgraphMatching.out",
    #     default_args=["-filter", "GQL", "-order", "GQL", "-engine", "LFTJ", "-num", "MAX"],
    #     timeout=120,
    # )
    # 
    # data_graph = "/home/wangshuo/resource/datasets/parler_data/dataset_one/ground_truth/data_graph/parler.graph"
    # query_graph = "/home/wangshuo/resource/datasets/parler_data/dataset_one/ground_truth/query_graph/query_dense_1_1.graph"
    # output_file = "/home/wangshuo/resource/datasets/parler_data/dataset_one/ground_truth/structure_match_gt.csv"
    # 
    # result = matcher.run(data_graph, query_graph, save_results=True, result_file=output_file)
    # 
    # print("匹配数 (#Embeddings):", result["parsed"].get("embeddings", "N/A"))
    # print("结果已保存到:", output_file)

    # 2.1.3 根据精确的子图匹配结果集，以及目标推理节点的oracle值，计算最终真实的答案ST_IF_Truth
    T_true = compute_T_true(
        gt_path="/home/wangshuo/resource/datasets/parler_data/dataset_one/ground_truth/result/query_dense_1_3.graph_matches.csv",
        id_mapping_path="/home/wangshuo/resource/datasets/parler_data/dataset_one/data_graph/id_mapping.csv",
        post_csv_path="/home/wangshuo/resource/datasets/parler_data/dataset_one/csv_data/post.csv",
        gt_match_col="u2",   # 精确匹配结果中代表 Post 节点的列
        prob_col="ML1_oracle1_probability",
        prob_threshold=0.5
    )
    # 3.处理 Fastest 输出结果，将估计值与目标节点的文件合并
    BASE_DIR = "/home/wangshuo/resource/datasets/parler_data/dataset_one/results"
    SV_FILE = os.path.join(BASE_DIR, "in_estimateW_result.txt")
    MAP_DIR = "/home/wangshuo/resource/datasets/parler_data/dataset_one/data_graph"
    MAP_FILE = os.path.join(MAP_DIR, "id_mapping.csv")
    POST_FILE = os.path.join('/home/wangshuo/resource/datasets/parler_data/dataset_one/csv_data', "post.csv")
    OUTPUT_FILE = os.path.join(BASE_DIR, "post_with_estimate.csv")

    merger = FastestEstimateMerger(
        sv_file=SV_FILE,
        map_file=MAP_FILE,
        post_file=POST_FILE,
        output_file=OUTPUT_FILE
    )
    merged_df = merger.run()
    print("\n[INFO] 运行完成，前 5 行预览：")
    print(merged_df.head())

    # 4. 解析 Fastest 输出结果： 基于单推理谓词也就是root_label各个节点的权重估计值进行proxy指导采样
    csv_path = "/home/wangshuo/resource/datasets/parler_data/dataset_one/results/post_with_estimate.csv"
    sampler = ProxyStratifiedSampler(csv_path,T_true=T_true)

    run_times = 3
    methods = {
        "proxy_importance": sampler.run_proxy_importance,
        "proxy_uniform": sampler.run_proxy_uniform,
        "proxyE_importance": sampler.run_proxyE_importance,
        "proxyE_uniform": sampler.run_proxyE_uniform
    }

    results = {}
    for name, func in methods.items():
        T_list, Q_list = [], []
        print(f"\nRunning {name} for {run_times} times ...")
        for i in range(run_times):
            out = func()
            T_list.append(out["T_hat"])
            Q_list.append(out["Qerror"])
        results[name] = {
            "T_hat_mean": np.mean(T_list),
            "Qerror_mean": np.mean(Q_list),
            "T_hat_std": np.std(T_list),
            "Qerror_std": np.std(Q_list)
        }

    print("\n=== 四种方法结果对比（10次平均） ===")
    print(f"{'Method':20s} | {'T_hat(mean±std)':25s} | {'Qerror(mean±std)':25s}")
    print("-" * 80)
    for name, res in results.items():
        print(f"{name:20s} | {res['T_hat_mean']:.3f} ± {res['T_hat_std']:.3f}      "
              f"| {res['Qerror_mean']:.4f} ± {res['Qerror_std']:.4f}")

V1.5:  单谓词多查询图的结构优先整体流程示例，包括：

In [ ]:
import os

import numpy as np

from pythonProject.src.Structure_first.fastest_pipeline import FastestGraphConverter, FastestEstimateMerger
from pythonProject.src.Structure_first.graph_sample import FastestRunner
from pythonProject.src.Structure_first.precision_submatching import ExactSubgraphMatcher
from pythonProject.src.Structure_first.proxy_sample import ProxyStratifiedSampler, compute_T_true

# 以下是单推理谓词的结构优先的整体流程示例

if __name__ == "__main__":
    datasets_name = "parler_data"
    dataset_name = "dataset_one"
    CSV_BASE_DIR = f"/home/wangshuo/resource/datasets/{datasets_name}/{dataset_name}/csv_data"
    Graph_Lib_Dir = f"/home/wangshuo/resource/datasets/{datasets_name}/{dataset_name}/data_graph"
    # # 1. 将 CSV 数据转换为 GraphLib 格式，这个格式中边是带标签的
    # converter = FastestGraphConverter(CSV_BASE_DIR,Graph_Lib_Dir)
    # converter.run()

    # 2.基于GraphLib数据，调用Fastest进行树采样或图采样，得到特定标签下所有或部分点的估计值
    # runner = FastestRunner(
    #     build_dir="/home/wangshuo/projects/FaSTest-main/build"
    # )
    # # # 默认执行 ./Fastest -d parler --ROOT_LABEL 1 (表示推理谓词所在节点的标签)
    # code, output = runner.run(dataset="parler", root_label=1)
    # 2.1同时调用准确的子图匹配算法得到真实结果的基数，这些准确子图匹配算法只能处理边不带标签的图
    # # 2.1.1 所以首先基于原来边带标签的图，精简成边不带标签的图
    # converter = FastestGraphConverter(CSV_BASE_DIR,Graph_Lib_Dir)
    # converter.simplify_graph_merge_edges_update_degree(input_path=Graph_Lib_Dir+'/parler.graph',
    #                                                    output_path='/home/wangshuo/resource/datasets/parler_data/dataset_one/ground_truth/data_graph/parler.graph')
    # # 2.1.2 然后对精简后的数据图调用准确子图匹配算法进行匹配，这里得到的GT对应的查询图，应该与Fastest对应的查询图等价（就算边没有标签也没有影响，且两点不存在多条边）
    # matcher = ExactSubgraphMatcher(
    #     exe_path="/home/wangshuo/projects/SubgraphMatching/build/matching/SubgraphMatching.out",
    #     default_args=["-filter", "GQL", "-order", "GQL", "-engine", "LFTJ", "-num", "MAX"],
    #     timeout=120,
    # )
    # 
    # data_graph = "/home/wangshuo/resource/datasets/parler_data/dataset_one/ground_truth/data_graph/parler.graph"
    # query_graph = "/home/wangshuo/resource/datasets/parler_data/dataset_one/ground_truth/query_graph/query_dense_1_1.graph"
    # output_file = "/home/wangshuo/resource/datasets/parler_data/dataset_one/ground_truth/structure_match_gt.csv"
    # 
    # result = matcher.run(data_graph, query_graph, save_results=True, result_file=output_file)
    # 
    # print("匹配数 (#Embeddings):", result["parsed"].get("embeddings", "N/A"))
    # print("结果已保存到:", output_file)

    # # 2.1.3 根据精确的子图匹配结果集，以及目标推理节点的oracle值，计算最终真实的答案ST_IF_Truth
    T_true = compute_T_true(
        gt_path="/home/wangshuo/resource/datasets/parler_data/dataset_one/ground_truth/result/query_dense_1_3.graph_matches.csv",
        id_mapping_path="/home/wangshuo/resource/datasets/parler_data/dataset_one/data_graph/id_mapping.csv",
        post_csv_path="/home/wangshuo/resource/datasets/parler_data/dataset_one/csv_data/post.csv",
        gt_match_col="u2",   # 精确匹配结果中代表 Post 节点的列
        prob_col="ML1_oracle1_probability",
        prob_threshold=0.5
    )
    # # # 3.处理 Fastest 输出结果，将估计值与目标节点的文件合并
    BASE_DIR = "/home/wangshuo/resource/datasets/parler_data/dataset_one/results"
    SV_FILE = os.path.join(BASE_DIR, "in_estimateW_result.txt")
    MAP_DIR = "/home/wangshuo/resource/datasets/parler_data/dataset_one/data_graph"
    MAP_FILE = os.path.join(MAP_DIR, "id_mapping.csv")
    POST_FILE = os.path.join('/home/wangshuo/resource/datasets/parler_data/dataset_one/csv_data', "post.csv")
    OUTPUT_FILE = os.path.join(BASE_DIR, "post_with_estimate.csv")

    merger = FastestEstimateMerger(
        sv_file=SV_FILE,
        map_file=MAP_FILE,
        post_file=POST_FILE,
        output_file=OUTPUT_FILE
    )
    merged_df = merger.run()
    print("\n[INFO] 运行完成，前 5 行预览：")
    print(merged_df.head())

    # # 4. 解析 Fastest 输出结果： 基于单推理谓词也就是root_label各个节点的权重估计值进行proxy指导采样
    csv_path = "/home/wangshuo/resource/datasets/parler_data/dataset_one/results/post_with_estimate.csv"
    sampler = ProxyStratifiedSampler(csv_path,T_true=T_true)

    run_times = 2
    methods = {
        "proxy_importance": sampler.run_proxy_importance,
        "proxy_uniform": sampler.run_proxy_uniform,
        "proxyE_importance": sampler.run_proxyE_importance,
        "proxyE_uniform": sampler.run_proxyE_uniform
    }

    results = {}
    for name, func in methods.items():
        T_list, Q_list = [], []
        print(f"\nRunning {name} for {run_times} times ...")
        for i in range(run_times):
            out = func()
            T_list.append(out["T_hat"])
            Q_list.append(out["Qerror"])
        results[name] = {
            "T_hat_mean": np.mean(T_list),
            "Qerror_mean": np.mean(Q_list),
            "T_hat_std": np.std(T_list),
            "Qerror_std": np.std(Q_list)
        }

    print("\n=== 四种方法结果对比（10次平均） ===")
    print(f"{'Method':20s} | {'T_hat(mean±std)':25s} | {'Qerror(mean±std)':25s}")
    print("-" * 80)
    for name, res in results.items():
        print(f"{name:20s} | {res['T_hat_mean']:.3f} ± {res['T_hat_std']:.3f}      "
              f"| {res['Qerror_mean']:.4f} ± {res['Qerror_std']:.4f}")

测试--- FastestEstimateMerger 多文件合并

多文件合并

In [ ]:
import os
import re
from typing import Optional

import pandas as pd


class FastestEstimateMerger:
    DEFAULT_RENAME_MAP = {
        "ML1_oracle1_probability": "post_oracle1",
        "ML1_proxy4b1_probability": "post_proxy4b1",
        "ML1_proxy2b1_probability": "post_proxy2b1",
        "orig_id": "postId"
    }

    def __init__(
        self,
        sv_file: str,
        map_file: str,
        post_file: str,
        output_file: Optional[str] = None,
        rename_map: Optional[dict] = None,
    ):
        self.sv_file = sv_file
        self.map_file = map_file
        self.post_file = post_file
        if output_file is None:
            base = os.path.dirname(sv_file) or "."
            output_file = os.path.join(base, "post_with_estimate.csv")
        self.output_file = output_file
        self.rename_map = rename_map or self.DEFAULT_RENAME_MAP

        # parsed dataframes
        self.idmap_df = None
        self.posts_map_df = None
        self.sv_df = None
        self.merged_map_sv = None
        self.posts_df = None
        self.result_df = None

    # -------------------------
    # Step 1: 读取 id 映射
    # -------------------------
    def load_idmap(self):
        if not os.path.exists(self.map_file):
            raise FileNotFoundError(f"id map not found: {self.map_file}")
        self.idmap_df = pd.read_csv(self.map_file, dtype=str, keep_default_na=False)
        # 尝试把 internal_id 转为 int（方便合并）
        if "internal_id" in self.idmap_df.columns:
            try:
                self.idmap_df["internal_id"] = self.idmap_df["internal_id"].astype(int)
            except Exception:
                # 如果失败则保留为字符串
                pass

        # 选择 Post 类型映射（有 type 列时）
        if "type" in self.idmap_df.columns:
            self.posts_map_df = self.idmap_df[self.idmap_df["type"] == "Post"][["internal_id", "orig_id"]].copy()
        else:
            # 没有 type，则保守地把所有映射当作 Post（退化处理）
            self.posts_map_df = self.idmap_df[["internal_id", "orig_id"]].copy()

        print(f"[INFO] 载入映射表，共 {len(self.idmap_df)} 条，Post 类型 {len(self.posts_map_df)} 条")

    # ============================================================
    # Step 2: 解析 sv_estimate_result.txt —— 支持多个 Query 块 (estimate_q1, estimate_q2, ...)
    # ============================================================
    def parse_sv_file(self):
        if not os.path.exists(self.sv_file):
            raise FileNotFoundError(f"sv estimate file not found: {self.sv_file}")

        pattern_line = re.compile(r"^(\d+)\s+([0-9eE\.\-NaNnan]+)$")
        queries_data = {}
        current_query = None
        current_records = []
        all_est_value = None
        query_idx = 0

        with open(self.sv_file, "r") as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                if line.startswith("Query:"):
                    # 如果已有上一个 query 的记录，则先保存它
                    if current_query is not None and current_records:
                        queries_data[f"estimate_q{query_idx}"] = pd.DataFrame(current_records)
                    query_idx += 1
                    current_query = line.split("Query:")[-1].strip()
                    current_records = []
                    all_est_value = None
                elif line.startswith("All Est:"):
                    try:
                        all_est_value = float(line.split("All Est:")[-1].strip())
                    except Exception:
                        all_est_value = None
                else:
                    m = pattern_line.match(line)
                    if m:
                        node_id, val = m.groups()
                        try:
                            est_val = float(val)
                        except Exception:
                            # 若 val 为非数字（例如 'nan'），设为 NaN -> later fillna(0) 可选
                            est_val = float("nan")
                        current_records.append({
                            "internal_id": int(node_id),
                            f"estimate_q{query_idx}": est_val,
                            "all_est": all_est_value
                        })

        # 保存最后一个 query 的记录
        if current_query is not None and current_records:
            queries_data[f"estimate_q{query_idx}"] = pd.DataFrame(current_records)

        print(f"[INFO] 检测到 {len(queries_data)} 个 Query 块")

        # 合并所有 query-dataframe，按 internal_id 对齐（outer 保留所有 internal_id）
        merged_all = None
        for name, df in sorted(queries_data.items()):
            if merged_all is None:
                merged_all = df
            else:
                merged_all = pd.merge(merged_all, df, on="internal_id", how="outer")

        if merged_all is None:
            # 如果没有任何记录
            self.sv_df = pd.DataFrame(columns=["internal_id"])
        else:
            # 用 0 填充 NaN（你需要保留每列，因此不要丢弃 NaN）
            merged_all = merged_all.sort_values("internal_id").reset_index(drop=True)
            # 只填充 estimate 列的 NaN 为 0（保留 all_est 列也可以）
            estimate_cols = [c for c in merged_all.columns if c.startswith("estimate_q")]
            for c in estimate_cols:
                merged_all[c] = pd.to_numeric(merged_all[c], errors="coerce").fillna(0.0)
            self.sv_df = merged_all

        print(f"[INFO] 合并后包含 {len([c for c in self.sv_df.columns if c.startswith('estimate_q')])} 个 estimate 列，共 {len(self.sv_df)} 条记录")

    # -------------------------
    # Step 3: 合并 id 映射（internal_id -> orig_id）
    # -------------------------
    def merge_idmap_with_sv(self):
        if self.sv_df is None:
            raise RuntimeError("sv_df 未解析，请先调用 parse_sv_file()")
        if self.posts_map_df is None:
            raise RuntimeError("idmap 未加载，请先调用 load_idmap()")

        left = self.posts_map_df.copy()
        right = self.sv_df.copy()

        # 统一类型以便合并
        if left["internal_id"].dtype == object:
            right["internal_id"] = right["internal_id"].astype(str)
        else:
            try:
                left["internal_id"] = left["internal_id"].astype(int)
            except Exception:
                left["internal_id"] = left["internal_id"].astype(str)
                right["internal_id"] = right["internal_id"].astype(str)

        merged = pd.merge(left, right, on="internal_id", how="inner")
        self.merged_map_sv = merged
        print(f"[INFO] 合并映射后：{len(merged)} 条有效 Post 节点")

    # -------------------------
    # Step 4: 合并原始 post.csv（保留所有 estimate_q* 列）
    # -------------------------
    def merge_with_posts(self):
        if not os.path.exists(self.post_file):
            raise FileNotFoundError(f"post file not found: {self.post_file}")
        self.posts_df = pd.read_csv(self.post_file, dtype=str, keep_default_na=False)
        id_col = None
        for candidate in ["id:ID", "id", "postId", "post_id"]:
            if candidate in self.posts_df.columns:
                id_col = candidate
                break
        if id_col is None:
            id_col = self.posts_df.columns[0]
            print(f"[WARN] 未找到常见的 id 列名，使用第一列: {id_col}")

        # 把 merged_map_sv 的 orig_id 与 posts_df 的 id 列对齐
        merged = pd.merge(self.posts_df, self.merged_map_sv, left_on=id_col, right_on="orig_id", how="left")
        self.result_df = merged

        # 说明：不对多个 estimate_q* 做聚合，直接保留；
        # 如果需要把 NaN -> 0（对后续统计与采样友好），可以在下面进行填充
        estimate_cols = [c for c in self.result_df.columns if c.startswith("estimate_q")]
        if estimate_cols:
            # 将这些列强制为数值型，NaN 转为 0，便于后续统计
            for c in estimate_cols:
                self.result_df[c] = pd.to_numeric(self.result_df[c], errors="coerce").fillna(0.0)

        print(f"[INFO] 合并 post 与估计结果后: 总 Post 行数={len(self.result_df)}，包含估计列: {estimate_cols}")

    # -------------------------
    # Step 5: 重命名 ML 列、保证 postId 存在，并做统计（不再过滤掉 estimate==0 的行）
    # === 修改点：filter_and_rename 改为 保留所有记录并重命名 / 输出每个 estimate 列的非零计数 ===
    # -------------------------
    def filter_and_rename(self):
        if self.result_df is None:
            raise RuntimeError("result_df 未准备好，请先调用 merge_with_posts()")

        # 保留全部记录（不要按 estimate>0 过滤）
        df = self.result_df.copy()

        # 重命名 ml 列（如果存在）
        available_rename = {k: v for k, v in self.rename_map.items() if k in df.columns}
        if available_rename:
            df.rename(columns=available_rename, inplace=True)

        # 确保有 postId 列（优先使用已重命名的 postId / orig_id，否则从 posts_df 的 id 列中构造）
        if "postId" not in df.columns:
            if "orig_id" in df.columns:
                df.rename(columns={"orig_id": "postId"}, inplace=True)
            else:
                for candidate in ["id:ID", "id", "postId", "post_id"]:
                    if candidate in self.posts_df.columns:
                        df["postId"] = df[candidate]
                        break

        # 统计每个 estimate_q* 的非零数量
        estimate_cols = [c for c in df.columns if c.startswith("estimate_q")]
        nonzero_stats = {}
        for c in estimate_cols:
            nonzero_count = int((pd.to_numeric(df[c], errors="coerce").fillna(0.0) > 0).sum())
            nonzero_stats[c] = nonzero_count

        # 保存回成员变量
        self.result_df = df
        self.estimate_cols = estimate_cols
        self.nonzero_stats = nonzero_stats

        print(f"[INFO] 完成重命名与 postId 检查。发现 {len(estimate_cols)} 个估计列：{estimate_cols}")
        for k, v in nonzero_stats.items():
            print(f"  -> {k}: nonzero_count = {v}")

    # -------------------------
    # Step 6: 保存结果并报告
    # -------------------------
    def save_and_report(self):
        if self.result_df is None:
            raise RuntimeError("result_df 未准备好，请先调用 filter_and_rename()")

        # 保存全部列（包含多个 estimate_q* 列）
        self.result_df.to_csv(self.output_file, index=False)
        print(f"[✅] 已生成合并文件: {self.output_file}")

        total_posts = len(self.result_df)
        print("\n===== 📊 统计结果 =====")
        print(f"总 Post 数量: {total_posts}")
        if hasattr(self, "estimate_cols") and self.estimate_cols:
            print(f"包含的估计列: {self.estimate_cols}")
            for c in self.estimate_cols:
                print(f"  {c}: 非零数 = {self.nonzero_stats.get(c, 0)}")
        else:
            print("未检测到 estimate_q* 列。")

    # -------------------------
    # 全流程 run
    # -------------------------
    def run(self):
        self.load_idmap()
        self.parse_sv_file()
        self.merge_idmap_with_sv()
        self.merge_with_posts()
        self.filter_and_rename()
        self.save_and_report()
        return self.result_df
